In [17]:
import cv2
import numpy as np
import math

# Load the video file
video_path = 'output_sequences/sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Use goodFeaturesToTrack to find feature points in the first frame
p0 = cv2.goodFeaturesToTrack(gray1, maxCorners=0, qualityLevel=0.3, minDistance=7)

# Create a mask for drawing purposes
mask = np.zeros_like(frame1)

# Initialize total distance
total_distance = 0

average_vx_list = []
average_vy_list = []
V_list = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow from the previous frame to the current frame
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]


    step = 65.5
    dt = step/1000 # dt in second fixed for all frames

    vx_list = []
    vy_list = []

    # Draw the tracks and calculate distances
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = np.int32(new.ravel())
        c, d = np.int32(old.ravel())

        # calculate the speed in x and y axis
        vx = abs((a-c)/dt)
        vy = abs((b-d)/dt)
        vx_list.append(vx)
        vy_list.append(vy)


        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)


    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)    
    V = math.sqrt(pow(average_vx,2) + pow(average_vy,2))
    
    average_vx_list.append(average_vx)
    average_vy_list.append(average_vy)
    V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()
    p0 = good_new.reshape(-1, 1, 2)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()


In [16]:
import pandas as pd
# Create a DataFrame
df = pd.DataFrame({'vx': average_vx_list, 'vy': average_vy_list, 'v': V_list})

In [17]:
df.head()

,vx,vy,v
0,103.301279,70.189353,124.890750
1,96.163379,65.034202,116.089805
2,50.560127,53.534252,73.635878
3,32.318826,45.999802,56.218220
4,17.677782,39.574126,43.342998


In [18]:
# Save the DataFrame to a CSV file
df.to_csv('Predicted_velocity/Lucas-Kanade/Shi-Tomasi.csv', index=False)

# harris corner detection

In [22]:
# Load the video file
video_path = 'output_sequences/sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Use Harris Corner Detector to find feature points in the first frame
dst = cv2.cornerHarris(gray1, blockSize=2, ksize=3, k=0.04)
# Normalize the output to get values between 0 and 255
dst_norm = cv2.normalize(dst, None, 0, 255, cv2.NORM_MINMAX)
# Threshold to get the corners
threshold = 0.01 * dst_norm.max()
corners = np.where(dst_norm > threshold)

# Limit the number of corners by selecting the top N corners based on Harris response
N = 100  # Adjust the number of corners as needed
top_corners = np.argsort(dst_norm[corners], axis=None)[-N:]
p0 = np.float32(list(zip(corners[1][top_corners], corners[0][top_corners]))).reshape(-1, 1, 2)

# Create a mask for drawing purposes
mask = np.zeros_like(frame1)

# Initialize total distance
total_distance = 0

average_vx_list = []
average_vy_list = []
V_list = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow from the previous frame to the current frame
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    step = 65.5
    dt = step / 1000  # dt in seconds fixed for all frames

    vx_list = []
    vy_list = []

    # Draw the tracks and calculate distances
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = np.int32(new.ravel())
        c, d = np.int32(old.ravel())

        # calculate the speed in x and y axis
        vx = abs((a - c) / dt)
        vy = abs((b - d) / dt)
        vx_list.append(vx)
        vy_list.append(vy)

        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)

    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)
    V = math.sqrt(pow(average_vx, 2) + pow(average_vy, 2))

    average_vx_list.append(average_vx)
    average_vy_list.append(average_vy)
    V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()
    p0 = good_new.reshape(-1, 1, 2)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()


In [23]:
# Create a DataFrame
df_haris = pd.DataFrame({'vx': average_vx_list, 'vy': average_vy_list, 'v': V_list})
df_haris.head()

,vx,vy,v
0,111.959288,75.873236,135.246553
1,90.831984,54.745933,106.054545
2,59.218136,44.722029,74.208136
3,38.399260,34.389699,51.547595
4,17.759776,26.483876,31.887385


In [24]:
# Save the DataFrame to a CSV file
df_haris.to_csv('Predicted_velocity/Lucas-Kanade/Harris.csv', index=False)

# FAST (Features from Accelerated Segment Test)

In [25]:
# Load the video file
video_path = 'output_sequences/sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Use FAST to find feature points in the first frame
fast = cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=True)
kp = fast.detect(gray1, None)
p0 = np.float32([kp[i].pt for i in range(len(kp))]).reshape(-1, 1, 2)

# Create a mask for drawing purposes
mask = np.zeros_like(frame1)

# Initialize total distance
total_distance = 0

average_vx_list = []
average_vy_list = []
V_list = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow from the previous frame to the current frame
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    step = 65.5
    dt = step / 1000  # dt in seconds fixed for all frames

    vx_list = []
    vy_list = []

    # Draw the tracks and calculate distances
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = np.int32(new.ravel())
        c, d = np.int32(old.ravel())

        # calculate the speed in x and y axis
        vx = abs((a - c) / dt)
        vy = abs((b - d) / dt)
        vx_list.append(vx)
        vy_list.append(vy)

        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)

    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)
    V = math.sqrt(pow(average_vx, 2) + pow(average_vy, 2))

    average_vx_list.append(average_vx)
    average_vy_list.append(average_vy)
    V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()
    
    # Use FAST to find feature points in the current frame
    kp = fast.detect(gray1, None)
    p0 = np.float32([kp[i].pt for i in range(len(kp))]).reshape(-1, 1, 2)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()


In [26]:
# Create a DataFrame
df_fast = pd.DataFrame({'vx': average_vx_list, 'vy': average_vy_list, 'v': V_list})
df_fast.head()

,vx,vy,v
0,111.990920,69.085678,131.585702
1,101.861475,73.016318,125.328140
2,49.585698,42.272838,65.159300
3,31.068953,38.349143,49.355208
4,33.957709,50.314233,60.701302


In [27]:
# Save the DataFrame to a CSV file
df_fast.to_csv('Predicted_velocity/Lucas-Kanade/FAST.csv', index=False)